# 01 — Data Audit
**Telco Customer Churn — Portfolio Project**

Goal of this notebook: understand what "good" means before training anything.
We load the raw data, validate its schema, quantify quality issues, and produce
a data dictionary and assumptions log.

> **Rule:** everything in this notebook runs on the *full raw dataset* only to
> **describe** it. No statistic computed here (mean, median, mode, category
> lists) is reused as a transformation at inference time — those live inside
> the scikit-learn `Pipeline` in `03_modeling.ipynb`, fit only on training folds.


In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np

from src.data import (
    load_raw_data, audit_data, clean_data, make_synthetic_telco_data,
    EXPECTED_COLUMNS, TARGET_COLUMN, ID_COLUMN,
)

pd.set_option("display.max_columns", 50)
RAW_PATH = "../data/raw/telco_churn.csv"


## 1. Load data

If you have downloaded the real Kaggle dataset
(https://www.kaggle.com/datasets/blastchar/telco-customer-churn), place the
CSV at `data/raw/telco_churn.csv` and the cell below will load it directly.

If the file is not present, we fall back to a **synthetic dataset** that
mirrors the exact schema and rough statistical shape of the real data, purely
so this notebook (and the rest of the project) can run end-to-end without
an internet connection. Swap in the real file for an actual submission.


In [2]:
import os

if not os.path.exists(RAW_PATH):
    print("Real Kaggle CSV not found — generating a synthetic demo dataset with the same schema.")
    os.makedirs("../data/raw", exist_ok=True)
    make_synthetic_telco_data(n_rows=5000).to_csv(RAW_PATH, index=False)

raw_df = load_raw_data(RAW_PATH)
raw_df.head()


2026-10-20 23:27:06,702 | INFO | Loaded raw data: 7043 rows x 21 columns from ..\data\raw\telco_churn.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Shape, dtypes, and unique counts

In [3]:
print(f"Shape: {raw_df.shape[0]} rows x {raw_df.shape[1]} columns")
raw_df.dtypes.to_frame("dtype")


Shape: 7043 rows x 21 columns


,dtype
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


In [4]:
unique_counts = raw_df.nunique().sort_values()
unique_counts.to_frame("n_unique")


,n_unique
gender,2
SeniorCitizen,2
Partner,2
Dependents,2
PhoneService,2
PaperlessBilling,2
Churn,2
MultipleLines,3
TechSupport,3
StreamingTV,3


## 3. customerID uniqueness

`customerID` is a primary-key-like identifier. It should never be used as a
model feature — it carries no generalizable signal, and a high-cardinality
categorical like this would just let a model memorize training rows (a
severe form of overfitting/leakage). We keep it only for joining predictions
back to a business-readable report.


In [5]:
n_duplicated_ids = raw_df[ID_COLUMN].duplicated().sum()
print(f"Duplicate customerIDs: {n_duplicated_ids}")
print(f"customerID is {'unique' if n_duplicated_ids == 0 else 'NOT unique'} per row.")


Duplicate customerIDs: 0
customerID is unique per row.


## 4. TotalCharges conversion

**What could go wrong if `TotalCharges` remains a string?**
Every downstream numeric operation (imputation, scaling, correlation, model
fitting) would either fail or silently treat the column as categorical,
destroying its ordinal information (e.g. \$20 vs \$8000).

We coerce it with `errors="coerce"`, which turns unparsable blank strings
into `NaN` rather than raising, so we can inspect exactly which rows are
affected before deciding how to handle them.


In [6]:
coerced = pd.to_numeric(raw_df["TotalCharges"], errors="coerce")
newly_missing = coerced.isna() & ~raw_df["TotalCharges"].isna()
print(f"Rows where TotalCharges becomes missing after coercion: {newly_missing.sum()}")

raw_df.loc[newly_missing, ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]


Rows where TotalCharges becomes missing after coercion: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


These rows are almost always customers with **`tenure == 0`** — brand-new
accounts that haven't been billed yet. This is a legitimate business
segment, not a data error. We will impute `TotalCharges` using the median
**inside the pipeline** (fit on training folds only), and separately confirm
this hypothesis below.


In [7]:
zero_tenure_and_missing = raw_df[(raw_df["tenure"] == 0)]
print(f"Zero-tenure rows: {len(zero_tenure_and_missing)}")
print(f"Of those, TotalCharges blank/unparsable: {newly_missing[raw_df['tenure']==0].sum()}")


Zero-tenure rows: 11
Of those, TotalCharges blank/unparsable: 11


## 5. Full data-quality audit (via `src.data.audit_data`)

In [8]:
report = audit_data(raw_df)

print(f"Rows: {report.n_rows}, Columns: {report.n_cols}")
print(f"Duplicate rows: {report.duplicate_row_count}")
print(f"Duplicate customerIDs: {report.duplicate_id_count}")
print(f"TotalCharges coerced to NaN: {report.total_charges_coerced_nulls}")
print(f"Zero-tenure rows: {report.zero_tenure_rows}")
print(f"Target prevalence (Churn = Yes): {report.target_prevalence:.2%}")


Rows: 7043, Columns: 21
Duplicate rows: 0
Duplicate customerIDs: 0
TotalCharges coerced to NaN: 11
Zero-tenure rows: 11
Target prevalence (Churn = Yes): 26.54%


### Data-quality table: issue, affected rows, decision, rationale

In [9]:
quality_table = report.to_frame()
quality_table


,issue,affected_rows,decision,rationale
0,TotalCharges stored as string with blank entries,11,Coerce to numeric with errors='coerce'; impute...,Blank values correspond to customers with zero...
1,Duplicate rows,0,None found,Exact duplicate rows would let the same custom...
2,customerID uniqueness,0,Retain customerID for reporting only; excluded...,customerID is an arbitrary identifier with no ...
3,Zero-tenure customers,11,"Keep, but flag as a small, plausible edge case",New customers naturally have tenure=0 and blan...


In [10]:
os.makedirs("../reports", exist_ok=True)
quality_table.to_csv("../reports/data_quality_report.csv", index=False)
print("Saved to reports/data_quality_report.csv")


Saved to reports/data_quality_report.csv


## 6. Missingness by column

In [11]:
missing = raw_df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing.to_frame("n_missing") if len(missing) else "No missing values detected in raw string form (TotalCharges blanks are strings, not NaN, until coerced)."


'No missing values detected in raw string form (TotalCharges blanks are strings, not NaN, until coerced).'

## 7. Data dictionary

| Column | Type | Description |
|---|---|---|
| customerID | id | Unique customer identifier (excluded from model features) |
| gender | categorical | Male / Female |
| SeniorCitizen | binary (0/1) | Whether the customer is a senior citizen |
| Partner | categorical | Has a partner (Yes/No) |
| Dependents | categorical | Has dependents (Yes/No) |
| tenure | numeric | Months the customer has stayed with the company |
| PhoneService | categorical | Has phone service |
| MultipleLines | categorical | Has multiple phone lines |
| InternetService | categorical | DSL / Fiber optic / No |
| OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies | categorical | Add-on services, dependent on InternetService |
| Contract | categorical | Month-to-month / One year / Two year |
| PaperlessBilling | categorical | Yes/No |
| PaymentMethod | categorical | Billing method |
| MonthlyCharges | numeric | Current monthly bill |
| TotalCharges | numeric | Cumulative billed amount (string with blanks in raw data) |
| **Churn** | **target** | Yes/No — did the customer leave |

## 8. Assumptions and risks log

- **Prediction timing**: we assume the model scores customers at a point in
  time using their *current* snapshot of tenure, contract, and billing
  features — not future information. All features here are plausibly known
  at prediction time; none are post-outcome artifacts.
- **Zero-tenure / blank TotalCharges**: treated as a legitimate new-customer
  segment, imputed via median inside the pipeline rather than dropped, to
  avoid discarding a real and important population (new customers are often
  a leading indicator group for early-churn intervention).
- **customerID**: excluded from features; retained only for the final report.
- **Class imbalance**: churn prevalence is roughly 20-27%; this motivates the
  imbalance-handling and threshold-tuning work in later notebooks — accuracy
  alone would be a misleading metric.
- **Synthetic data caveat** (if used): this demo dataset is a schema-accurate
  stand-in for the real Kaggle file. Any concrete numbers in the audit/EDA
  reflect the synthetic generator, not real telecom customers. Replace with
  the actual Kaggle CSV before drawing real business conclusions.

## Exit criterion
✅ Every feature is documented, every cleaning decision is justified, and the
business objective (predict churn, act on it cost-effectively) is stated.
